# CIFAR-100 Relative-Damping Test: d=0.5 vs d=1

This notebook runs a controlled **75-epoch CIFAR-100 / ViT-S/16** comparison between:

- the new candidate, trace-relative damping `d=0.5`;
- the validated local anchor, trace-relative damping `d=1.0`.

`d=0.5` applies about `sqrt(0.5)=0.707` times each factor mean eigenvalue as damping. It is the less-damped candidate and previously looked stronger early, but it has never been compared with `d=1` under this exact final schedule.

Every other optimizer, model, augmentation, and schedule setting is held fixed. The anchor is deliberately rerun in the same Colab session so GPU model, precision fallback, package versions, and data-loader environment cannot masquerade as a damping effect.

Expected runtime is roughly 1.5--3 hours per run on an A100/L4 and longer on a T4. Partial summaries are synchronized to Google Drive while training.


## 1. Configuration

Use a GPU runtime. The paper-comparable path uses BF16 and therefore prefers an A100, L4, or another GPU with native BF16 support. On a T4, the notebook can fall back to FP16 training plus FP32 inverse-factor computation; those results are useful for within-notebook screening but should not be mixed directly with the BF16 paper runs.


In [ ]:
# Values are ordered so the previously untested candidate runs first.
D_VALUES = [0.5, 1.0]
SWEEP_NAME = "low_pair"
SEED = 12
EPOCHS = 75
BATCH_SIZE = 128
WORKERS = 2

REPO_URL = "https://github.com/LukeTri/kfac-muon.git"
REPO_REF = "main"  # Change this if the relative-damping implementation is on another branch.

MOUNT_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/kfac-muon-colab"
ALLOW_FP16_FALLBACK = True
KEEP_FINAL_CHECKPOINT = False  # Summaries/args are enough for selecting d.
SYNC_SECONDS = 20

# Set to a subset to rerun only one configuration, for example [0.5].
print("D values:", D_VALUES)


## 2. Mount Drive and inspect the GPU

The output directory is shared by both damping notebooks. This lets either notebook's final analysis cell plot every completed run.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_OUTPUT_ROOT = Path(DRIVE_ROOT) / "cifar100_d_sweep" / SWEEP_NAME
LOCAL_OUTPUT_ROOT = Path("/content/kfac-cifar100-runs")
DATA_ROOT = Path("/content/data/cifar100")
SOURCE_ROOT = Path("/content/data/torchvision")

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
print("Persistent results:", DRIVE_OUTPUT_ROOT)


In [ ]:
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU before continuing."
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
bf16_ok = bool(torch.cuda.is_bf16_supported())
print("PyTorch:", torch.__version__)
print("GPU:", gpu_name, "capability:", capability, "BF16:", bf16_ok)

if bf16_ok:
    AMP_DTYPE = "bfloat16"
    INVERSE_DTYPE = "bfloat16"
    PRECISION_LABEL = "bf16"
else:
    if not ALLOW_FP16_FALLBACK:
        raise RuntimeError("This GPU lacks native BF16. Request an A100/L4 or enable FP16 fallback.")
    AMP_DTYPE = "float16"
    INVERSE_DTYPE = "float32"
    PRECISION_LABEL = "fp16_fallback"
    print("WARNING: using FP16 training and FP32 inverse factors. Compare only against the d=1 anchor from this notebook.")


## 3. Clone the repository and install dependencies

The guard at the end intentionally fails if the selected Git ref predates the trace-relative damping and norm-matching implementation. This is safer than silently running a different optimizer.


In [ ]:
REPO_ROOT = Path("/content/kfac-muon")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF], check=True)
if REPO_REF == "main":
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", "main"], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_ROOT / "requirements.txt"), "matplotlib", "pandas"
], check=True)

train_text = (REPO_ROOT / "train.py").read_text()
required_markers = [
    "--kfac-damping-mode",
    "relative_trace",
    "--kfac-match-muon-norm-ratio",
    "--kfac-reference-only",
]
missing = [marker for marker in required_markers if marker not in train_text]
if missing:
    raise RuntimeError(
        "The selected Git ref does not contain the current relative-damping implementation. "
        f"Missing: {missing}. Push/select the updated branch before running the sweep."
    )

print("Repository commit:")
subprocess.run(["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"], check=True)


## 4. Prepare CIFAR-100

This materializes CIFAR-100 in ImageFolder format under `/content`. It is done once per Colab VM and does not consume Drive space.


In [ ]:
data_script = REPO_ROOT / "scripts/data/download_cifar100_vast.sh"
if not data_script.exists():
    data_script = REPO_ROOT / "download_cifar100_vast.sh"
if not data_script.exists():
    raise FileNotFoundError("Could not locate the CIFAR-100 preparation script.")

env = os.environ.copy()
env.update({
    "SOURCE_ROOT": str(SOURCE_ROOT),
    "OUT_ROOT": str(DATA_ROOT),
    "OVERWRITE": "0",
})
subprocess.run(["bash", str(data_script)], cwd=REPO_ROOT, env=env, check=True)

train_images = sum(1 for p in (DATA_ROOT / "train").rglob("*.png"))
val_images = sum(1 for p in (DATA_ROOT / "val").rglob("*.png"))
print("train images:", train_images, "val images:", val_images)
assert train_images == 50_000 and val_images == 10_000


## 5. Training helper

The helper uses the final clean CIFAR recipe:

- ViT-S/16, batch size 128, seed 12;
- 75 epochs, 8 warmup epochs, cosine LR `3e-3 -> 1e-5`;
- Mixup/CutMix 0.2, disabled after epoch 56;
- norm ratio `r=1`, factor EMA 0.95, statistics/factor cadence 2;
- patch embedding and classifier head included.

Training writes locally for speed. `summary.csv` and `args.yaml` are copied to Drive every 20 seconds and after completion. By default the large checkpoint is not retained.


In [ ]:
import csv
import re
import signal
import time


def d_to_tag(value):
    return (f"{float(value):g}").replace(".", "p")


def experiment_name(d):
    return (
        f"vits16_c100_kfac_reltrace_d{d_to_tag(d)}_r1_"
        f"lr3e3_e{EPOCHS}_s{SEED}_colab_{PRECISION_LABEL}"
    )


def last_completed_epoch(summary_path):
    path = Path(summary_path)
    if not path.exists() or path.stat().st_size == 0:
        return -1
    with path.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    valid = [int(float(row["epoch"])) for row in rows if row.get("epoch", "").replace(".", "", 1).isdigit()]
    return max(valid, default=-1)


def sync_small_artifacts(local_run, persistent_run):
    persistent_run.mkdir(parents=True, exist_ok=True)
    for name in ("summary.csv", "args.yaml"):
        src = local_run / name
        if src.exists():
            tmp = persistent_run / f".{name}.tmp"
            shutil.copy2(src, tmp)
            os.replace(tmp, persistent_run / name)


def build_command(d, name):
    return [
        sys.executable, "-u", "train.py",
        "--data-dir", str(DATA_ROOT),
        "--dataset", "image_folder",
        "--train-split", "train",
        "--val-split", "val",
        "--num-classes", "100",
        "--model", "vit_small_patch16_224",
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--validation-batch-size", str(BATCH_SIZE),
        "--workers", str(WORKERS),
        "--opt", "kfac_muon",
        "--lr", "3e-3",
        "--opt-betas", "0.9", "0.95",
        "--weight-decay", "0.07",
        "--sched", "cosine",
        "--warmup-epochs", "8",
        "--warmup-lr", "1e-5",
        "--min-lr", "1e-5",
        "--mixup", "0.2",
        "--cutmix", "0.2",
        "--mixup-off-epoch", "56",
        "--smoothing", "0.1",
        "--reprob", "0.1",
        "--drop-path", "0.1",
        "--seed", str(SEED),
        "--log-interval", "200",
        "--val-interval", "1",
        "--checkpoint-hist", "1",
        "--checkpoint-final-only",
        "--output", str(LOCAL_OUTPUT_ROOT),
        "--experiment", name,
        "--amp",
        "--amp-dtype", AMP_DTYPE,
        "--kfac-damping-mode", "relative_trace",
        "--kfac-damping", str(d),
        "--kfac-match-muon-norm-ratio", "1.0",
        "--kfac-momentum", "0.9",
        "--kfac-nesterov",
        "--kfac-muon-eps", "0.038",
        "--kfac-muon-lr-adjustment", "match_rms_adamw",
        "--kfac-stats-update-every", "2",
        "--kfac-factor-update-every", "2",
        "--kfac-ema-decay", "0.95",
        "--no-kfac-lm-adapt-damping",
        "--kfac-track-muon-reference",
        "--no-kfac-exclude-first-last",
        "--kfac-aux-no-decay",
        "--kfac-use-inverse-factors",
        "--kfac-inverse-compute-dtype", INVERSE_DTYPE,
    ]


def run_one(d):
    name = experiment_name(d)
    local_run = LOCAL_OUTPUT_ROOT / name
    persistent_run = DRIVE_OUTPUT_ROOT / name
    persistent_summary = persistent_run / "summary.csv"

    if last_completed_epoch(persistent_summary) >= EPOCHS - 1:
        print(f"[skip] d={d}: already complete at {persistent_run}")
        return

    # An interrupted run retains its partial summary on Drive, but restarts cleanly.
    if local_run.exists():
        shutil.rmtree(local_run)

    command = build_command(d, name)
    print("\n" + "=" * 90)
    print(f"Starting d={d}: {name}")
    print(" ".join(command))
    print("=" * 90 + "\n")

    process = subprocess.Popen(command, cwd=REPO_ROOT)
    try:
        while process.poll() is None:
            sync_small_artifacts(local_run, persistent_run)
            time.sleep(SYNC_SECONDS)
    except KeyboardInterrupt:
        print("Interrupt received; asking train.py to stop cleanly...")
        process.send_signal(signal.SIGINT)
        process.wait(timeout=120)
    finally:
        sync_small_artifacts(local_run, persistent_run)

    if process.returncode not in (0, None):
        raise RuntimeError(f"Training failed for d={d} with return code {process.returncode}")

    if KEEP_FINAL_CHECKPOINT:
        candidates = sorted(local_run.glob("*.pth.tar"), key=lambda p: p.stat().st_mtime)
        if candidates:
            shutil.copy2(candidates[-1], persistent_run / candidates[-1].name)

    final_epoch = last_completed_epoch(persistent_summary)
    print(f"Finished d={d}; persistent summary through epoch {final_epoch}: {persistent_summary}")

    if final_epoch >= EPOCHS - 1 and not KEEP_FINAL_CHECKPOINT:
        shutil.rmtree(local_run, ignore_errors=True)


## 6. Run the pair

The candidate is first. If you only have time for one run, stop after it finishes; the `d=1` run is the same-environment control.


In [ ]:
for damping in D_VALUES:
    run_one(float(damping))


## 7. Compare all available damping runs

This cell discovers summaries produced by either notebook in the shared Drive folder. It reports endpoints and plots validation top-1, validation loss, and the pointwise top-1 gap relative to `d=1` when that anchor is available.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

summary_files = sorted(DRIVE_OUTPUT_ROOT.glob("vits16_c100_kfac_reltrace_d*_r1_*/summary.csv"))
if not summary_files:
    raise FileNotFoundError(f"No summaries found under {DRIVE_OUTPUT_ROOT}")

runs = {}
records = []
for path in summary_files:
    match = re.search(r"_d([0-9p]+)_r1_", path.parent.name)
    if not match:
        continue
    d = float(match.group(1).replace("p", "."))
    frame = pd.read_csv(path)
    frame = frame[pd.to_numeric(frame["epoch"], errors="coerce").notna()].copy()
    frame["epoch"] = frame["epoch"].astype(int)
    for col in ("eval_top1", "eval_loss", "train_loss"):
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    runs[d] = frame
    final = frame.iloc[-1]
    best_idx = frame["eval_top1"].idxmax()
    records.append({
        "d": d,
        "epochs present": len(frame),
        "final epoch": int(final["epoch"]),
        "best top-1": float(frame.loc[best_idx, "eval_top1"]),
        "best epoch": int(frame.loc[best_idx, "epoch"]),
        "final top-1": float(final["eval_top1"]),
        "final val loss": float(final["eval_loss"]),
        "final train loss": float(final["train_loss"]),
    })

results_table = pd.DataFrame(records).sort_values("d").reset_index(drop=True)
display(results_table.style.format({
    "d": "{:.2g}",
    "best top-1": "{:.3f}",
    "final top-1": "{:.3f}",
    "final val loss": "{:.4f}",
    "final train loss": "{:.4f}",
}))
results_table.to_csv(DRIVE_OUTPUT_ROOT / "d_sweep_results.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for d, frame in sorted(runs.items()):
    axes[0].plot(frame["epoch"], frame["eval_top1"], label=f"d={d:g}", linewidth=2)
    axes[1].plot(frame["epoch"], frame["eval_loss"], label=f"d={d:g}", linewidth=2)
axes[0].set(title="Validation top-1", xlabel="Epoch", ylabel="Top-1 (%)")
axes[1].set(title="Validation loss", xlabel="Epoch", ylabel="Loss")

if 1.0 in runs:
    anchor = runs[1.0].set_index("epoch")["eval_top1"]
    for d, frame in sorted(runs.items()):
        if d == 1.0:
            continue
        series = frame.set_index("epoch")["eval_top1"]
        common = series.index.intersection(anchor.index)
        axes[2].plot(common, series.loc[common] - anchor.loc[common], label=f"d={d:g} - d=1", linewidth=2)
    axes[2].axhline(0, color="black", linewidth=1)
    axes[2].set(title="Top-1 gap vs d=1", xlabel="Epoch", ylabel="Percentage points")
else:
    axes[2].text(0.5, 0.5, "Run d=1 to show paired gaps", ha="center", va="center")
    axes[2].set_axis_off()

for ax in axes[:2]:
    ax.grid(alpha=0.2)
    ax.legend()
if 1.0 in runs:
    axes[2].grid(alpha=0.2)
    axes[2].legend()
fig.tight_layout()
figure_path = DRIVE_OUTPUT_ROOT / "d_sweep_curves.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved table:", DRIVE_OUTPUT_ROOT / "d_sweep_results.csv")
print("Saved figure:", figure_path)


## Interpretation checklist

Prefer a value of `d` only if it improves the late trajectory or endpoint without an obvious validation-loss regression. Because `r=1` fixes each layer's KFAC step norm to the plain-Muon reference, differences primarily reflect the preconditioned **direction**, not a larger update. Do not select solely from the first 20--30 epochs: our earlier `d=0.5` run looked stronger early but weaker late under a confounded schedule.
